In [11]:
import cv2
import mediapipe as mp
import numpy as np
import json

# --- Load calibration data ---
with open("finger_calibration.json", "r") as f:
    calibration = json.load(f)

# Convert np.float64 to normal float
for finger, vals in calibration.items():
    calibration[finger]['min'] = float(vals['min'])
    calibration[finger]['max'] = float(vals['max'])

# --- MediaPipe setup ---
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)
smooth_bend = {f: 0 for f in calibration.keys()}

def calculate_angle(a, b, c):
    """Calculate angle between 3 points (joint coordinates)."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    return np.degrees(np.arccos(cosine_angle))

def get_finger_angles(hand_landmarks):
    """Return dict of angles for each finger."""
    lm = hand_landmarks.landmark
    return {
        'Thumb': calculate_angle(
            [lm[2].x, lm[2].y], [lm[3].x, lm[3].y], [lm[4].x, lm[4].y]),
        'Index': calculate_angle(
            [lm[5].x, lm[5].y], [lm[6].x, lm[6].y], [lm[8].x, lm[8].y]),
        'Middle': calculate_angle(
            [lm[9].x, lm[9].y], [lm[10].x, lm[10].y], [lm[12].x, lm[12].y]),
        'Ring': calculate_angle(
            [lm[13].x, lm[13].y], [lm[14].x, lm[14].y], [lm[16].x, lm[16].y]),
        'Pinky': calculate_angle(
            [lm[17].x, lm[17].y], [lm[18].x, lm[18].y], [lm[20].x, lm[20].y])
    }

# --- Main capture loop ---
with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb_frame)

        if result.multi_hand_landmarks:
            for hand_landmarks in result.multi_hand_landmarks:
                # Draw hand
                mp_drawing.draw_landmarks(
                    frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                # Calculate angles
                finger_angles = get_finger_angles(hand_landmarks)

                # Normalize and smooth
                for finger, current_angle in finger_angles.items():
                    min_a, max_a = calibration[finger]['min'], calibration[finger]['max']
                    bent_percent = np.clip(
                        (current_angle - min_a) / (max_a - min_a), 0, 1) * 100
                    smooth_bend[finger] = 0.7 * smooth_bend[finger] + 0.3 * bent_percent

                # Display on screen
                y = 40
                for f, val in smooth_bend.items():
                    cv2.putText(frame, f"{f}: {val:.1f}%", (30, y),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (60, 255, 60), 2)
                    y += 30

        cv2.imshow("Finger Bend Tracker", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


FileNotFoundError: [Errno 2] No such file or directory: 'finger_calibration.json'

In [2]:
import cv2
import mediapipe as mp
import numpy as np
import json
import math
import time

# --- Load calibration ---
with open("finger_calibration.json", "r") as f:
    calibration = json.load(f)

# Convert np.float64 → float
for finger, vals in calibration.items():
    calibration[finger]['min'] = float(vals['min'])
    calibration[finger]['max'] = float(vals['max'])

# --- Mediapipe setup ---
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

# --- Helper: compute angle between 3 points ---
def calc_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

# --- Finger landmark triplets (for each joint) ---
finger_joints = {
    "Thumb":  [(1, 2, 4)],
    "Index":  [(5, 6, 8)],
    "Middle": [(9, 10, 12)],
    "Ring":   [(13, 14, 16)],
    "Pinky":  [(17, 18, 20)]
}

# --- Initialize smoothing state ---
smoothed = {f: 0 for f in finger_joints.keys()}

# --- Capture loop ---
cap = cv2.VideoCapture(0)
with mp_hands.Hands(min_detection_confidence=0.6, min_tracking_confidence=0.6, max_num_hands=1) as hands:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        h, w, _ = frame.shape

        if result.multi_hand_landmarks:
            for handLms in result.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, handLms, mp_hands.HAND_CONNECTIONS)
                landmarks = [(lm.x * w, lm.y * h, lm.z * w) for lm in handLms.landmark]

                for finger, triplets in finger_joints.items():
                    # average of angles for multiple joints (thumb only 1)
                    angles = [calc_angle(landmarks[a], landmarks[b], landmarks[c]) for a, b, c in triplets]
                    angle = np.mean(angles)

                    cal = calibration[finger]
                    min_a, max_a = cal['min'], cal['max']

                    # Normalize to 0–100
                    bent = np.clip((min_a - angle) / (min_a - max_a), 0, 1) * 100

                    # Smooth transitions
                    smoothed[finger] = 0.8 * smoothed[finger] + 0.2 * bent

                # Display on screen
                for i, (finger, val) in enumerate(smoothed.items()):
                    cv2.putText(frame, f"{finger}: {val:5.1f}%", (20, 40 + i * 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (50, 255, 255), 2)

        cv2.imshow("Finger Bend Detection", frame)


        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord('q'):  # 27 = ESC key
            break


cap.release()
cv2.destroyAllWindows()
